import os
import cv2
import pandas as pd
import numpy as np

from lxml import etree
import numpy as np
import os
from skimage import io
from skimage.transform import resize
from sklearn.model_selection import train_test_split
from PIL import Image
import matplotlib.pyplot as plt

In [2]:
# current path is fetched
current_path = os.getcwd()


In [6]:
# specifying classes and making df with Cat codes
filter = ['background','aeroplane','bicycle','bird','boat','bottle',
           'bus','car','cat','chair','cow',
           'diningtable','dog','horse','motorbike','person',
           'pottedplant','sheep','sofa','train','tvmonitor']

df_cats = pd.DataFrame(list(filter), 
               columns =['bbx_0_name'])
df_cats['bbx_0_name'] = df_cats['bbx_0_name'].astype('category')
df_cats['Categories'] = df_cats['bbx_0_name'].cat.codes+1
df_cats

,bbx_0_name,Categories
0,background,2
1,aeroplane,1
2,bicycle,3
3,bird,4
4,boat,5
5,bottle,6
6,bus,7
7,car,8
8,cat,9
9,chair,10


In [5]:
#### get image names from train.txt, val.txt and test.txt for classification task #####

image_dir = os.path.join(current_path,'VOCtrainval_11-May-2009','VOCdevkit','VOC2009')

image_list = open('segm_train.txt')
imagenames_train = list()
images_train = list()
for line in image_list:
    imagenames_train.append(line.rstrip())    
    file = os.path.join(image_dir, 'JPEGImages', line.strip()+'.jpg')
    img = cv2.imread(file)
    img = cv2.resize(img,(224,224))
    images_train.append(img) 
df_train = pd.DataFrame(list(zip(imagenames_train, images_train)), 
               columns =['fileID', 'Image'])    

image_list = open('segm_val.txt')
imagenames_val = list()
images_val = list()
for line in image_list:
    imagenames_val.append(line.rstrip())    
    file = os.path.join(image_dir, 'JPEGImages', line.strip()+'.jpg')
    img = cv2.imread(file)
    img = cv2.resize(img,(224,224))
    images_val.append(img) 
df_val = pd.DataFrame(list(zip(imagenames_val, images_val)), 
               columns =['fileID', 'Image']) 

image_list = open('segm_test.txt')
imagenames_test = list()
images_test = list()
for line in image_list:
    imagenames_test.append(line.rstrip())
    file = os.path.join(image_dir, 'JPEGImages', line.strip()+'.jpg')
    img = cv2.imread(file)
    img = cv2.resize(img,(224,224))
    images_test.append(img) 
df_test = pd.DataFrame(list(zip(imagenames_test, images_test)), 
               columns =['fileID', 'Image'])     

In [17]:
## functions for generating colourmaps for corresponding labels
COLORMAP = [[0, 0, 0], [128, 0, 0], [0, 128, 0], [128, 128, 0], [0, 0, 128],
            [128,0,128], [0,128,128], [128,128,128], [64,0,0], [192,0,0],
            [64,128,0], [192,128,0], [64,0,128], [192,0,128],
            [64,128,128], [192,128,128], [0,64,0], [128,64,0],
            [0,192,0], [128,192,0], [0,64,128]]

def build_colormap2label():
    """Build an RGB color to label mapping for segmentation."""
    colormap2label = np.zeros(256 ** 3)
    for i, colormap in enumerate(VOC_COLORMAP):
        colormap2label[(colormap[0]*256 + colormap[1])*256 + colormap[2]] = i
    return colormap2label


def voc_label_indices(colormap, colormap2label):
    """Map an RGB color to a label."""
    colormap = colormap.astype(np.int32)
    idx = ((colormap[:, :, 0] * 256 + colormap[:, :, 1]) * 256
           + colormap[:, :, 2])
    return colormap2label[idx]

CM2CLASS = np.zeros(256 ** 3) # every pixel 0~255，RGB 3 channel
for i,cm in enumerate(COLORMAP):
    CM2CLASS[(cm[0] * 256 + cm[1]) * 256 + cm[2]] = i # build index


In [19]:
### functions to contruct segmentation datasets in the form of np arrays


img_dir  = os.path.join(current_path,'VOCtrainval_11-May-2009','VOCdevkit','VOC2009','JPEGImages')
seg_dir = os.path.join(current_path,'VOCtrainval_11-May-2009','VOCdevkit','VOC2009','SegmentationClass')

## reading images from train, test and val lists and forming np arrays
def build_segmentation_dataset(seg_file):
    with open(seg_file) as file:
        lines = file.read().splitlines()
        train_filter = [line.strip() for line in lines]

    image_folder = os.path.join(current_path,'VOCtrainval_11-May-2009','VOCdevkit','VOC2009','JPEGImages')
    image_filenames = [os.path.join(image_folder, file) for f in train_filter for file in os.listdir(image_folder) if
                       f in file]
    x = np.array([resize(io.imread(img_f), (224, 224, 3)) for img_f in image_filenames]).astype('float32')

    seg_folder = os.path.join(current_path,'VOCtrainval_11-May-2009','VOCdevkit','VOC2009','SegmentationClass')
    image_filenames = [os.path.join(seg_folder, file) for f in train_filter for file in os.listdir(seg_folder) if
                       f in file]
    y = np.array([get_target(img_f) for img_f in image_filenames])

    return x, y


## transforming colour image to 3-d one-hot encoding array
def get_target(img_path):
    label_im = Image.open(img_path).convert('RGB')
    im = label_im.resize((224,224), Image.ANTIALIAS)
    data = np.array(im, dtype='int32')
    idx = (data[:, :, 0] * 256 + data[:, :, 1]) * 256 + data[:, :, 2]
    label_map =  np.array(CM2CLASS[idx], dtype='int64') # 2-d class map, class of every pixel

    return label_map


## saving arrays in npy files
def get_segmentation_dataset():
    train_file = os.path.join(current_path,'segm_train.txt')
    x_train, y_train = build_segmentation_dataset(train_file)
    print('%i training images number' % (x_train.shape[0]))
    print(x_train.shape)
    print(y_train.shape)
    np.save('./x_seg_train_.npy', x_train)
    np.save('./y_seg_train_.npy', y_train)

    val_file = os.path.join(current_path,'segm_val.txt')
    x_val, y_val = build_segmentation_dataset(val_file)
    print('%i validation images number' % (x_val.shape[0]))
    print(x_val.shape)
    print(y_val.shape)
    np.save('./x_seg_val.npy', x_val)
        np.save('./y_seg_val.npy', y_val)

    test_file = os.path.join(current_path,'segm_test.txt')
    x_test, y_test = build_segmentation_dataset(test_file)
    print('%i validation images number' % (x_test.shape[0]))
    print(x_test.shape)
    print(y_test.shape)
    np.save('./x_seg_test.npy', x_test)
    np.save('./y_seg_test.npy', y_test)
    
get_segmentation_dataset()

1049 training images number
(1049, 224, 224, 3)
(1049, 224, 224)
301 validation images number
(301, 224, 224, 3)
(301, 224, 224)
149 validation images number
(149, 224, 224, 3)
(149, 224, 224)


In [ ]:
x_train = np.load('./x_seg_train_.npy') 
x_val = np.load("./x_seg_val.npy")
x_test = np.load("./x_seg_test.npy")

y_train = np.load('./y_seg_train_.npy') 
y_val = np.load("./y_seg_val.npy")
# y_test = np.load("./y_seg_test.npy")

In [ ]:
from keras.utils import to_categorical
y_train = to_categorical(y_train)
y_val = to_categorical(y_val)

In [ ]:
## specifying directories for Annotation xmls and JPEG images
dir_anno = os.path.join(current_path,'VOCtrainval_11-May-2009','VOCdevkit','VOC2009','Annotations')
img_dir  = os.path.join(current_path,'VOCtrainval_11-May-2009','VOCdevkit','VOC2009','JPEGImages')
seg_dir = os.path.join(current_path,'VOCtrainval_11-May-2009','VOCdevkit','VOC2009','SegmentationClass')


In [ ]:
## using pretrained vgg16 without top layers

from keras.preprocessing.image import load_img
from keras.preprocessing.image import img_to_array
from keras.applications.vgg16 import preprocess_input
from keras.applications.vgg16 import decode_predictions
from keras.applications.vgg16 import VGG16
from keras.layers import Flatten, Dense,Conv2D,Dropout,UpSampling2D
from keras.models import Model
from keras.layers.normalization import BatchNormalization
from keras.layers import concatenate
from keras.optimizers import Adam
import tensorflow as tf
from keras import backend as K
from keras.losses import categorical_crossentropy

In [ ]:
model = VGG16(include_top= False, input_shape=(224,224,3))

# # specifying additional layers
# # The dense layer: Flatten() accepts the output from the last layer of the base model as an input to the new model 
x = Flatten()(model.layers[-1].output)
# # next, a dense layer with softmax activation is connected. number of classes is set as 2
x = Dense(4096, activation='relu')(x)
x = Dense(21, activation = 'softmax',name='classification')(x)

# # full model is now specified alongwith the pre-specified input and expected output 
model = Model(inputs = model.input, outputs = x)
model.summary()

# # compiling model
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])


for layer in model.layers[:-3]:
    layer.trainable = False

In [ ]:
image_size = 224
class_num = 21
learning_rate = 0.0001
epochs = 20

In [ ]:
def unet_upblock(filter,input,concat,concat_encoder):
    up_conv = BatchNormalization()(Conv2D(filters=filter,kernel_size=(2,2),activation='relu',padding='same',
                                          kernel_initializer='he_normal')(UpSampling2D(size=(2,2),interpolation='bilinear')(input)))
    if concat:
        merge = concatenate([concat_encoder,up_conv])
    else:
        merge = up_conv
    conv1 = BatchNormalization()(Conv2D(filters=filter,kernel_size=(3,3),activation='relu',padding='same',
                                        kernel_initializer='he_normal')(merge))
    conv2 = BatchNormalization()(Conv2D(filters=filter, kernel_size=(3, 3), activation='relu', padding='same',
                                        kernel_initializer='he_normal')(conv1))

    return conv2


def unet(image_feature,vgg):
    conv1 = Conv2D(filters=1024,kernel_size=(3,3),activation='relu',padding='same',kernel_initializer='he_normal')(image_feature)
    conv2 = Conv2D(filters=1024, kernel_size=(3,3), activation='relu', padding='same', kernel_initializer='he_normal')(conv1)
    decoder_input = Dropout(0.5)(conv2)

    decoder1 = unet_upblock(filter=512,input=decoder_input,concat_encoder=vgg.get_layer('block5_conv3').output,concat=True)
    decoder2 = unet_upblock(filter=256,input=decoder1,concat_encoder=vgg.get_layer('block4_conv3').output,concat=True)
    decoder3 = unet_upblock(filter=128,input=decoder2,concat_encoder=vgg.get_layer('block3_conv2').output,concat=False)
    decoder4 = unet_upblock(filter=64,input=decoder3,concat_encoder=vgg.get_layer('block2_conv2').output,concat=False)
    decoder5 = unet_upblock(filter=64, input=decoder4, concat_encoder=vgg.get_layer('block1_conv2').output,concat=False)

    result = Conv2D(filters=class_num,kernel_size=(1,1),activation='relu',padding='same',
                    kernel_initializer='he_normal',name='seg_output')(decoder5)

    return result

def softmax_sparse_crossentropy(y_true, y_pred):
    y_pred = K.reshape(y_pred, (-1, K.int_shape(y_pred)[-1]))
    log_softmax = tf.nn.log_softmax(y_pred)

    y_true = K.one_hot(tf.to_int32(K.flatten(y_true)), K.int_shape(y_pred)[-1])
    unpacked = tf.unstack(y_true, axis=-1)
    y_true = tf.stack(unpacked[:], axis=-1)

    cross_entropy = -K.sum(y_true * log_softmax, axis=1)
    cross_entropy_mean = K.mean(cross_entropy)

    return cross_entropy_mean

In [ ]:
vgg = VGG16(weights='imagenet',include_top = False,input_shape=(image_size,image_size,3))
image_feature = vgg.output
result = unet(image_feature,vgg)

model = Model(vgg.input, result)
model.summary()

model.compile(optimizer=Adam(lr=learning_rate,decay=learning_rate/epochs),loss='categorical_crossentropy',metrics=['accuracy'])
model.fit(x=x_train,y=y_train,batch_size=16,epochs=epochs,validation_split=0.1,validation_data=(x_val,y_val))

In [ ]:
model.save('Object_segmentation_02052020_20epochs_RV.h5')

In [ ]:
from sklearn.metrics import accuracy_score
y_pred = model.predict(x_test)
y_pred =np.argmax(y_pred, axis=1)
print("Accuracy: {0:0.1f}%".format(accuracy_score(y_test,y_pred)*100))